In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, mean_squared_error



In [20]:
df = pd.read_csv("data/data_join.csv")

print(df.shape)
df.head()

(9319, 13)


,LATITUDE,LONGITUDE,DATE,TOTALAL_KALINITY,ELECTRICAL_CONDUCTANCE,DISSOLVED_REACTIVE_PHOSPHORUS,PET,NIR,GREEN,SWIR16,SWIR22,NDMI,MNDWI
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0,174.2,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0,124.1,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0,127.5,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0,129.7,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0,129.2,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683


In [21]:
df["DATE"] = pd.to_datetime(df["DATE"], format="%d-%m-%Y")

df["month"] = df["DATE"].dt.month
df["year"] = df["DATE"].dt.year
df["dayofyear"] = df["DATE"].dt.dayofyear

In [22]:
features = [
    "LATITUDE",
    "LONGITUDE",
    "PET",
    "NIR",
    "GREEN",
    "SWIR16",
    "SWIR22",
    "NDMI",
    "MNDWI",
    "month",
    "dayofyear"
]

targets = [
    "TOTALAL_KALINITY",
    "ELECTRICAL_CONDUCTANCE",
    "DISSOLVED_REACTIVE_PHOSPHORUS"
]


X = df[features]
y = df[targets]

In [23]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)

pred = dummy.predict(X_test)


r2 = r2_score(y_test, pred)

print("Dummy R2:", r2)
print("Dummy RMSE:", np.sqrt(mean_squared_error(y_test, pred)))

Dummy R2: -0.00028815444235119614
Dummy RMSE: 206.42168679872344


In [24]:
for i in range(0, 3):
    r2 = r2_score(y_test, pred)

    print("Dummy R2:", r2)

Dummy R2: -0.00028815444235119614
Dummy R2: -0.00028815444235119614
Dummy R2: -0.00028815444235119614


In [25]:
from sklearn.model_selection import KFold


kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores = []

for train_idx, test_idx in kf.split(X):

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    dummy = DummyRegressor(strategy="mean")
    dummy.fit(X_train, y_train)

    pred = dummy.predict(X_test)

    r2 = r2_score(y_test, pred)

    r2_scores.append(r2)

print("CV R2:", np.mean(r2_scores))

CV R2: -0.00043391561800814785


In [32]:

model = LGBMRegressor(
    n_estimators=200,
    random_state=42,
    verbose=-1
)

for i in targets:
    y = df[i]

    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="r2"
    )

    print("Model CV R2:", scores.mean())

Model CV R2: 0.6915515525368605
Model CV R2: 0.7539666718351301
Model CV R2: 0.43078054595176124
